# CITS4012 A2: GPU runner (Kaggle / Colab)

This notebook runs every GPU job for the project and packs the results into **one zip** to download.
It is only a runner. The submission notebook is built separately from the same code.

**What it runs**
- the frozen main model (v1, soft targets), 3 seeds
- ablations A1–A5 and B2, 3 seeds each
- baselines B3 (BERTweet), B4 (t5-small) and B5 (Qwen2.5-1.5B)

Everything is evaluated on validation only; the test set stays locked. Expected time: 1–1.5 h on Kaggle's 2×T4.

### Kaggle (recommended)
1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then pick `gpu_runner.ipynb`.
2. Right panel → **Add Input → Upload** a dataset containing `train.csv` and `test.csv` (from the TweetQA folder). Any dataset name is fine.
3. Right panel → **Settings**: *Accelerator* = **GPU T4 x2**, *Internet* = **On** (needs a phone-verified account).
4. Click **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the tab.
5. Wait until the version is marked **complete** (Kaggle can take a while to save outputs after the code finishes). Then open the version → **Output** → download **`a2_gpu_results.zip`** and send it back.

### Colab
1. colab.research.google.com → **File → Upload notebook** → `gpu_runner.ipynb`.
2. **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all**. When asked, upload `train.csv` and `test.csv`.
4. Keep the tab open (free Colab can disconnect when idle). The zip downloads automatically at the end.


In [ ]:
# ---------------- settings ----------------
RUN_MAIN = True        # the training jobs listed at the top of this notebook
RUN_BASELINES = True  # B3 BERTweet, B4 t5-small, B5 Qwen2.5-1.5B (val)
RUN_STAGE34 = False    # also redo the Stage 3/4 sweep on GPU (12 more runs, ~30 min)

import os, sys, subprocess, glob, shutil, json, time
from pathlib import Path
ON_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
# On Kaggle the work happens in /tmp: everything in /kaggle/working gets uploaded as output when the
# version is saved, and a large output there made Kaggle hang. Only the final zip goes to /kaggle/working.
WORK = Path("/tmp/A2") if ON_KAGGLE else Path("/content/A2") if IN_COLAB else Path("A2_gpu").resolve()
(WORK / "src" / "baselines").mkdir(parents=True, exist_ok=True)
(WORK / "scripts").mkdir(exist_ok=True)
(WORK / "data").mkdir(exist_ok=True)
(WORK / "results" / "logs").mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gensim", "emoji", "nltk", "sentencepiece"], check=True)
import torch, gensim, transformers
N_GPU = torch.cuda.device_count()
assert N_GPU > 0, "No GPU: switch the accelerator/runtime to a GPU (see instructions above)"
print("platform:", "kaggle" if ON_KAGGLE else "colab" if IN_COLAB else "other", "| GPUs:",
      [torch.cuda.get_device_name(i) for i in range(N_GPU)], "| torch", torch.__version__, "| transformers", transformers.__version__, "| gensim", gensim.__version__)


In [ ]:
# ---------------- data: find train.csv / test.csv ----------------
def find(name):
    pats = [f"/kaggle/input/**/{name}", f"/content/**/{name}", str(WORK / "data" / name)]
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
for name in ("train.csv", "test.csv"):
    src = find(name)
    if src is None and IN_COLAB:
        from google.colab import files
        print(f"Please upload {name}")
        up = files.upload()
        src = next(iter(up))
    assert src, f"{name} not found: add it as a Kaggle input (see instructions)"
    if Path(src).resolve() != (WORK / "data" / name).resolve():
        shutil.copy(src, WORK / "data" / name)
import hashlib
for name in ("train.csv", "test.csv"):
    h = hashlib.sha1((WORK / "data" / name).read_bytes()).hexdigest()
    print(name, h)
assert hashlib.sha1((WORK / "data" / "train.csv").read_bytes()).hexdigest() == "8a52bb7dff56b11fdad1ab6fd3b848fbadc22849", "unexpected train.csv"
assert hashlib.sha1((WORK / "data" / "test.csv").read_bytes()).hexdigest() == "e33faa7614839d10a06b3607e74e51b70a0ee329", "unexpected test.csv"


## Step 1 — Write the project code (written to files; identical to `A2/src`)

In [ ]:
%%writefile src/__init__.py
# package marker


In [ ]:
%%writefile src/baselines/__init__.py
# package marker


In [ ]:
%%writefile src/evaluate.py
"""Evaluation metrics for TweetQA.

Every metric is computed per example as the max over the gold answers,
and corpus scores are the mean x 100.

- BLEU-1: nltk sentence_bleu, weights (1,0,0,0), smoothing method1
- METEOR: nltk meteor_score (needs WordNet)
- ROUGE-L: our own LCS implementation, F-measure with beta = 1.2 (the
  coco-caption convention used by the official TweetQA evaluation)
- EM and token-F1: SQuAD-style, computed on normalised text
"""
import re
import string
import unicodedata
from collections import Counter

METRICS = ("bleu1", "meteor", "rougeL", "em", "f1")
_ARTICLES = re.compile(r"\b(a|an|the)\b")
_nltk_ready = False


def _ensure_nltk():
    global _nltk_ready
    if _nltk_ready:
        return
    import nltk
    for pkg, path in [("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4")]:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)
    _nltk_ready = True


def normalize_answer(s):
    """Lowercase, replace ASCII and Unicode punctuation with spaces,
    drop articles, collapse whitespace."""
    s = s.lower()
    s = "".join(" " if (ch in string.punctuation or unicodedata.category(ch).startswith("P")) else ch
                for ch in s)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


def norm_tokens(s):
    return normalize_answer(s).split()


# ---------------------------------------------------------------- metrics

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred, gold):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    prec, rec = overlap / len(p), overlap / len(g)
    return 2 * prec * rec / (prec + rec)


def lcs_len(a, b):
    """Length of the longest common subsequence of two token lists (O(len(a)*len(b)) DP)."""
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            cur[j] = prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1])
        prev = cur
    return prev[-1]


def rouge_l(pred, gold, beta=1.2):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g and bool(p))
    lcs = lcs_len(p, g)
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(g)
    return (1 + beta ** 2) * prec * rec / (rec + beta ** 2 * prec)


def bleu1(pred, gold):
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return sentence_bleu([g], p, weights=(1, 0, 0, 0),
                         smoothing_function=SmoothingFunction().method1)


def meteor(pred, gold):
    _ensure_nltk()
    from nltk.translate.meteor_score import meteor_score
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return meteor_score([g], p)


_FUNCS = {"bleu1": bleu1, "meteor": meteor, "rougeL": rouge_l, "em": exact_match, "f1": token_f1}


def score_example(pred, golds, metrics=METRICS):
    """Per-example scores in [0, 1]: max over gold answers."""
    pred = pred or ""
    return {m: max(_FUNCS[m](pred, g) for g in golds) for m in metrics}


def evaluate(preds, golds_list, metrics=METRICS, return_per_example=False):
    """preds: list[str]; golds_list: list[list[str]]. Returns corpus scores (mean x 100)."""
    assert len(preds) == len(golds_list), (len(preds), len(golds_list))
    per = [score_example(p, g, metrics) for p, g in zip(preds, golds_list)]
    n = max(1, len(per))
    corpus = {m: 100.0 * sum(s[m] for s in per) / n for m in metrics}
    return (corpus, per) if return_per_example else corpus


def fmt(scores):
    return " ".join(f"{m}={scores[m]:.2f}" for m in METRICS if m in scores)


In [ ]:
%%writefile src/data.py
"""TweetQA data pipeline: loading, splits, offset-preserving tokenisation,
tweet normalisation, token flags, silver-span labelling, vocabularies and
a padded DataLoader.

Design notes (see results/DATA_REPORT.md):
- test.csv is the original TweetQA dev set. It stays locked (TEST_LOCKED)
  until the final evaluation. Validation is 10% of train, grouped by tweet.
- Every token keeps its character offsets into the raw tweet, so a
  predicted span (i, j) maps back to tweet[start_i:end_j].
- The normaliser only changes the *word* used for the embedding lookup.
  The char-CNN always sees the raw (lowercased) token, so @handles stay
  distinguishable after @user -> <user>.
"""
import ast
import csv
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset

try:
    from .evaluate import normalize_answer, rouge_l
except ImportError:  # inlined in the notebook / run as a script
    from evaluate import normalize_answer, rouge_l

TEST_LOCKED = True
DATA_DIR = Path(os.environ.get("TWEETQA_DIR", Path(__file__).resolve().parents[1] / "data"
                               if "__file__" in globals() else "data"))
SPLIT_SEED = 4012
VAL_FRAC = 0.10

# ------------------------------------------------------------------ loading

_ANSWER_ITEM = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"", re.S)


def parse_answers(field):
    """The Answer column is a numpy-array repr: ['a' 'b'] (space/newline separated, mixed quotes)."""
    items = [ast.literal_eval(m) for m in _ANSWER_ITEM.findall(field)]
    if not items:
        raise ValueError(f"unparseable answer field: {field!r}")
    return [str(a) for a in items]


def _read_csv(name):
    with open(Path(DATA_DIR) / name, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [dict(row_id=i, qid=r["qid"], question=r["Question"], tweet=r["Tweet"],
                 answers=parse_answers(r["Answer"]), answer_field=r["Answer"])
            for i, r in enumerate(rows)]


def _question_word(q):
    toks = tokenize(q.lower())
    w = toks[0][0] if toks else ""
    return w if w in {"what", "who", "how", "where", "why", "which", "when"} else "other"


def get_splits(seed=SPLIT_SEED, val_frac=VAL_FRAC):
    """Train/val from train.csv: exact duplicates dropped, grouped by tweet,
    stratified by the first question word of each tweet's first question."""
    rows, seen = [], set()
    for r in _read_csv("train.csv"):
        key = (r["question"], r["answer_field"], r["tweet"])
        if key not in seen:
            seen.add(key)
            rows.append(r)
    by_tweet = defaultdict(list)
    for r in rows:
        by_tweet[r["tweet"]].append(r)
    strata = defaultdict(list)
    for tweet in sorted(by_tweet):
        strata[_question_word(by_tweet[tweet][0]["question"])].append(tweet)
    rng = random.Random(seed)
    val_tweets = set()
    for s in sorted(strata):
        tweets = strata[s][:]
        rng.shuffle(tweets)
        val_tweets.update(tweets[:round(len(tweets) * val_frac)])
    train = [r for r in rows if r["tweet"] not in val_tweets]
    val = [r for r in rows if r["tweet"] in val_tweets]
    return train, val


def load_test():
    if TEST_LOCKED:
        raise RuntimeError("Test set is locked (TEST_LOCKED=True). It is used once, in Stage 8.")
    return _read_csv("test.csv")


# ------------------------------------------------------------- tokenisation

_URL = r"https?://\S+|www\.\S+"
_EMOJI_CHARS = "\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍"
_TOKEN_RE = re.compile(
    rf"(?P<url>{_URL})"
    r"|(?P<mention>@\w+)"
    r"|(?P<hashtag>#\w+)"
    rf"|(?P<emoji>[{_EMOJI_CHARS}]+)"
    r"|(?P<number>\d+(?:[.,:/]\d+)*)"
    r"|(?P<word>\w+(?:['’]\w+)*)"
    r"|(?P<sym>[^\w\s])",
    re.UNICODE,
)
_ELONG = re.compile(r"(.)\1{2,}")


def tokenize(text):
    """Returns [(raw_token, start, end, kind)] with char offsets into text."""
    return [(m.group(0), m.start(), m.end(), m.lastgroup) for m in _TOKEN_RE.finditer(text)]


def normalise_tokens(raw_tokens, use_normaliser=True):
    """Map raw tokens to model tokens: dicts with word, raw, start, end, kind.

    Normaliser: @user -> <user>, url -> <url>, emoji -> <emoji>,
    #tag -> '#' + 'tag', elongation squashed to 2 repeats, lowercase.
    With use_normaliser=False the word is just the lowercased raw token.
    """
    out = []
    for raw, s, e, kind in raw_tokens:
        if not use_normaliser:
            out.append(dict(word=raw.lower(), raw=raw, start=s, end=e, kind=kind))
        elif kind == "mention":
            out.append(dict(word="<user>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "url":
            out.append(dict(word="<url>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "emoji":
            out.append(dict(word="<emoji>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "hashtag":
            out.append(dict(word="#", raw="#", start=s, end=s + 1, kind=kind))
            tag = raw[1:]
            out.append(dict(word=_ELONG.sub(r"\1\1", tag.lower()), raw=tag, start=s + 1, end=e, kind=kind))
        else:
            out.append(dict(word=_ELONG.sub(r"\1\1", raw.lower()), raw=raw, start=s, end=e, kind=kind))
    return out


_BYLINE_RE = re.compile(r"\(@\w+\)\s+[A-Za-z]+\.?\s+\d{1,2},\s+\d{4}\s*$")
_BOUNDARY = {".", "!", "?", "—", "–", "-", "\"", "”", "|", ":"}


def byline_start(text, tokens):
    """Index of the first token of the trailing 'Name (@handle) Month D, YYYY'
    byline, or len(tokens) if there is none. The name is taken back to the
    previous sentence boundary, at most 6 tokens."""
    m = _BYLINE_RE.search(text)
    if not m:
        return len(tokens)
    k = next((i for i, t in enumerate(tokens) if t["start"] >= m.start()), len(tokens))
    j = k
    while j > 0 and k - j < 6 and tokens[j - 1]["raw"] not in _BOUNDARY:
        j -= 1
    return j


FLAG_NAMES = ("mention", "hashtag", "url", "emoji", "number", "capitalised", "in_byline", "in_other_seq")


def token_flags(tokens, other_words, byline_from=None):
    by = len(tokens) if byline_from is None else byline_from
    flags = []
    for i, t in enumerate(tokens):
        flags.append([
            float(t["kind"] == "mention"), float(t["kind"] == "hashtag"), float(t["kind"] == "url"),
            float(t["kind"] == "emoji"), float(t["kind"] == "number"),
            float(t["raw"][:1].isupper()), float(i >= by),
            float(t["word"] in other_words and t["word"] not in _STOP),
        ])
    return flags


_STOP = set("""a an the of to in on at for by with from and or but is are was were be been being am
do does did has have had i you he she it we they me him her us them my your his its our their this that
these those what who whom whose which when where why how not no yes so if than then there here as just
about into out up down over can will would should could may might must 's ? . , ! ' " - — : ; ( )""".split())


# ------------------------------------------------------------- silver spans

def _norm_toks(raw):
    return normalize_answer(raw).split()


def silver_spans(tokens, tweet, answers, max_len=15, tau=0.5):
    """Score every span (i, j), j - i < max_len, that starts and ends on a
    content token, by token-F1 against each gold answer (max over golds).

    Returns (best, best_f1, soft):
      best  : (i, j) with the highest F1, ties broken by ROUGE-L then length
      soft  : [(i, j, w)] for all spans with F1 >= tau, weights proportional to F1
    """
    ntok = [_norm_toks(t["raw"]) for t in tokens]
    content = [bool(x) for x in ntok]
    golds = [Counter(_norm_toks(a)) for a in answers]
    glens = [sum(g.values()) for g in golds]
    scored = []
    for i in range(len(tokens)):
        if not content[i]:
            continue
        bag, n = Counter(), 0
        for j in range(i, min(len(tokens), i + max_len)):
            bag.update(ntok[j])
            n += len(ntok[j])
            if not content[j]:
                continue
            f = 0.0
            for g, gl in zip(golds, glens):
                if gl == 0:
                    continue
                ov = sum((bag & g).values())
                if ov:
                    p, r = ov / n, ov / gl
                    f = max(f, 2 * p * r / (p + r))
            if f > 0:
                scored.append((f, i, j))
    if not scored:
        return None, 0.0, []
    top = max(f for f, _, _ in scored)
    ties = [(i, j) for f, i, j in scored if f == top]
    if len(ties) > 1:
        def key(ij):
            txt = tweet[tokens[ij[0]]["start"]:tokens[ij[1]]["end"]]
            return (max(rouge_l(txt, a) for a in answers), -(ij[1] - ij[0]), -ij[0])
        ties.sort(key=key, reverse=True)
    soft = [(i, j, f) for f, i, j in scored if f >= tau]
    z = sum(w for _, _, w in soft)
    soft = [(i, j, w / z) for i, j, w in soft]
    return ties[0], top, soft


def span_text(ex, i, j):
    return ex["tweet"][ex["t_tokens"][i]["start"]:ex["t_tokens"][j]["end"]]


# ------------------------------------------------------------------- vocab

class Vocab:
    PAD, UNK = "<pad>", "<unk>"

    def __init__(self, counter, min_freq=2, specials=()):
        self.itos = [self.PAD, self.UNK] + [s for s in specials if s not in (self.PAD, self.UNK)]
        self.itos += sorted([w for w, c in counter.items() if c >= min_freq and w not in self.itos],
                            key=lambda w: (-counter[w], w))
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def __call__(self, w):
        return self.stoi.get(w, 1)


SPECIALS = ("<user>", "<url>", "<emoji>")
MAX_WORD_CHARS = 16


def build_vocabs(examples, min_freq=2, char_min_freq=5):
    """Vocabularies come from the training split only."""
    wc, cc = Counter(), Counter()
    for ex in examples:
        for t in ex["t_tokens"] + ex["q_tokens"]:
            wc[t["word"]] += 1
            cc.update(t["raw"].lower()[:MAX_WORD_CHARS])
    return Vocab(wc, min_freq, SPECIALS), Vocab(cc, char_min_freq)


# ----------------------------------------------------------------- encoding

# tau: an example is trainable if its best silver span has F1 >= tau.
# soft_tau: minimum F1 for a span to join the soft target set (None = same as tau). Kept separate so the
# soft-set width can be varied without changing which examples are trained on.
DEFAULT_PREP = dict(use_normaliser=True, max_span_len=15, tau=0.5, soft_tau=None)


def preprocess(rows, prep=None):
    """Tokenise, normalise, flag and silver-label raw rows."""
    prep = {**DEFAULT_PREP, **(prep or {})}
    out = []
    for r in rows:
        t_tokens = normalise_tokens(tokenize(r["tweet"]), prep["use_normaliser"])
        q_tokens = normalise_tokens(tokenize(r["question"]), prep["use_normaliser"])
        by = byline_start(r["tweet"], t_tokens)
        t_words = {t["word"] for t in t_tokens}
        q_words = {t["word"] for t in q_tokens}
        soft_tau = prep["tau"] if prep.get("soft_tau") is None else prep["soft_tau"]
        best, best_f1, soft = silver_spans(t_tokens, r["tweet"], r["answers"], prep["max_span_len"], soft_tau)
        out.append(dict(
            r, t_tokens=t_tokens, q_tokens=q_tokens, byline_from=by,
            t_flags=token_flags(t_tokens, q_words, by), q_flags=token_flags(q_tokens, t_words),
            best=best, best_f1=best_f1, soft=soft,
            trainable=best is not None and best_f1 >= prep["tau"],
        ))
    return out


def encode(examples, wvocab, cvocab):
    for ex in examples:
        for side in ("t", "q"):
            toks = ex[f"{side}_tokens"]
            ex[f"{side}_ids"] = [wvocab(t["word"]) for t in toks]
            ex[f"{side}_chars"] = [[cvocab(c) for c in t["raw"].lower()[:MAX_WORD_CHARS]] or [1] for t in toks]
    return examples


class QADataset(Dataset):
    def __init__(self, examples, train=False):
        # during training, only examples with a usable silver span contribute to the loss
        self.examples = [e for e in examples if e["trainable"]] if train else list(examples)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def _pad_side(batch, side, n_flags):
    B = len(batch)
    T = max(1, max(len(ex[f"{side}_ids"]) for ex in batch))
    W = max(1, max((len(c) for ex in batch for c in ex[f"{side}_chars"]), default=1))
    words = torch.zeros(B, T, dtype=torch.long)
    chars = torch.zeros(B, T, W, dtype=torch.long)
    flags = torch.zeros(B, T, n_flags)
    mask = torch.zeros(B, T, dtype=torch.bool)
    for b, ex in enumerate(batch):
        n = len(ex[f"{side}_ids"])
        words[b, :n] = torch.tensor(ex[f"{side}_ids"], dtype=torch.long)
        for k, cs in enumerate(ex[f"{side}_chars"]):
            chars[b, k, :len(cs)] = torch.tensor(cs, dtype=torch.long)
        if n:
            flags[b, :n] = torch.tensor(ex[f"{side}_flags"])
        mask[b, :n] = True
    return words, chars, flags, mask


def collate(batch):
    tw, tc, tf, tm = _pad_side(batch, "t", len(FLAG_NAMES))
    qw, qc, qf, qm = _pad_side(batch, "q", len(FLAG_NAMES))
    B, T = tw.shape
    y_start = torch.full((B,), -1, dtype=torch.long)
    y_end = torch.full((B,), -1, dtype=torch.long)
    soft_start = torch.zeros(B, T)
    soft_end = torch.zeros(B, T)
    for b, ex in enumerate(batch):
        if ex["best"] is not None:
            y_start[b], y_end[b] = ex["best"]
        for i, j, w in ex["soft"]:
            soft_start[b, i] += w
            soft_end[b, j] += w
    return dict(t_words=tw, t_chars=tc, t_flags=tf, t_mask=tm,
                q_words=qw, q_chars=qc, q_flags=qf, q_mask=qm,
                y_start=y_start, y_end=y_end, soft_start=soft_start, soft_end=soft_end,
                meta=batch)


def make_loader(examples, batch_size=32, train=False, seed=0):
    ds = QADataset(examples, train=train)
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=train, collate_fn=collate, generator=g)


# ------------------------------------------------------------------ oracle

def oracle_predictions(examples):
    """Best silver span text for every example ('' if there is no overlapping span)."""
    return [span_text(ex, *ex["best"]) if ex["best"] is not None else "" for ex in examples]


In [ ]:
%%writefile src/runlog.py
"""Append-only experiment log: every reported number must come from a row in results/runs.csv."""
import csv
import json
import os
import time
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("A2_RESULTS", Path(__file__).resolve().parents[1] / "results"
                                  if "__file__" in globals() else "results"))
RUNS_CSV = RESULTS_DIR / "runs.csv"
FIELDS = ["run_id", "timestamp", "model", "config", "seed", "split",
          "bleu1", "meteor", "rougeL", "em", "f1", "train_time_s", "notes"]


def log_run(model, config, seed, split, scores, train_time_s=0.0, run_id=None, notes=""):
    RUNS_CSV.parent.mkdir(parents=True, exist_ok=True)
    new = not RUNS_CSV.exists()
    run_id = run_id or f"{model}-{split}-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    row = dict(run_id=run_id, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), model=model,
               config=json.dumps(config, sort_keys=True, default=str), seed=seed, split=split,
               train_time_s=round(train_time_s, 1), notes=notes,
               **{m: round(scores[m], 4) for m in ("bleu1", "meteor", "rougeL", "em", "f1")})
    with open(RUNS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    return run_id


def read_runs():
    if not RUNS_CSV.exists():
        return []
    with open(RUNS_CSV, encoding="utf-8") as f:
        return list(csv.DictReader(f))


In [ ]:
%%writefile src/model.py
"""TweetBiDAF: an extractive QA model for tweets, trained from scratch.

  1. Embedding: word embedding ⊕ char-CNN over the *raw* token ⊕ token flags
     -> 2-layer highway                                    (A1: use_char_cnn, A2: use_token_flags)
  2. Shared BiLSTM contextual encoder for tweet (H) and question (U)
  3. Trilinear similarity S_ij = w^T [h_i; u_j; h_i ∘ u_j]
     C2Q = softmax_j(S) U,  Q2C = softmax_i(max_j S) H
     G = [h; c2q; h ∘ c2q; h ∘ q2c]                        (A4: biattn_mode "c2q" drops Q2C)
  4. Gated multi-head self-attention over the projected G,
     residual + LayerNorm                                   (A3: use_self_attn)
  5. 2-layer modelling BiLSTM, start head, 1 more BiLSTM layer, end head,
     padding masks, best (i, j) with i <= j < i + L

The design follows BiDAF (Seo et al., 2017) with gated self-attention in the
style of R-Net (Wang et al., 2017). The tweet-specific parts are written for
this project: a char-CNN on raw tokens with a normalised word vocabulary,
token flags (including the byline flag), soft silver-span training, and
self-attention for long-range links between the tweet body and its byline.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

NEG = -1e30

DEFAULT_MODEL = dict(
    word_dim=100, char_dim=32, char_filters=50, char_widths=(3, 4, 5), n_flags=8,
    hidden=128, dropout=0.2, attn_heads=4, max_span_len=15,
    use_char_cnn=True, use_token_flags=True, use_self_attn=True, biattn_mode="both",
)


class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim, filters, widths):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.convs = nn.ModuleList(nn.Conv1d(char_dim, filters, w, padding=w // 2) for w in widths)
        self.out_dim = filters * len(widths)

    def forward(self, chars):                      # (B, T, W)
        B, T, W = chars.shape
        x = self.emb(chars.view(B * T, W)).transpose(1, 2)            # (BT, c, W)
        cmask = (chars.view(B * T, 1, W) != 0)
        outs = []
        for conv in self.convs:
            y = F.relu(conv(x))[:, :, :W]
            y = y.masked_fill(~cmask, 0.0)          # ReLU output >= 0, so masked max-pool is safe
            outs.append(y.max(dim=2).values)
        return torch.cat(outs, -1).view(B, T, -1)


class Highway(nn.Module):
    def __init__(self, dim, n_layers=2):
        super().__init__()
        self.lin = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))
        self.gate = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))

    def forward(self, x):
        for lin, gate in zip(self.lin, self.gate):
            g = torch.sigmoid(gate(x))
            x = g * F.relu(lin(x)) + (1 - g) * x
        return x


def reverse_padded(x, mask):
    """Reverse each sequence within its own length; padding stays at the end."""
    T = x.size(1)
    lengths = mask.sum(1, keepdim=True)                                  # (B, 1)
    t = torch.arange(T, device=x.device).unsqueeze(0)                    # (1, T)
    idx = torch.where(t < lengths, lengths - 1 - t, t)                   # (B, T)
    return x.gather(1, idx.unsqueeze(-1).expand_as(x))


class BiLSTM(nn.Module):
    """BiLSTM over right-padded batches, padding-exact without packing.

    The forward LSTM reads valid tokens before any padding, so its outputs at
    valid positions are exact. The backward direction runs a forward LSTM over
    each sequence reversed *within its length*, then un-reverses the output.
    The result equals a packed BiLSTM, but runs ~4x faster on MPS.
    """

    def __init__(self, in_dim, hidden, layers=1, dropout=0.0):
        super().__init__()
        self.fw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.bw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        for l, (fw, bw) in enumerate(zip(self.fw, self.bw)):
            if l > 0:
                x = self.drop(x)
            f = fw(x)[0]
            b = reverse_padded(bw(reverse_padded(x, mask))[0], mask)
            x = torch.cat([f, b], -1) * mask.unsqueeze(-1)
        return x


class GatedSelfAttention(nn.Module):
    """Multi-head self-attention, then a sigmoid gate on [x; attended], then residual + LayerNorm."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.gate = nn.Linear(2 * dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        attended, w = self.mha(x, x, x, key_padding_mask=~mask,
                               need_weights=True, average_attn_weights=True)
        g = torch.sigmoid(self.gate(torch.cat([x, attended], -1)))
        return self.norm(x + self.drop(g * attended)), w


class TweetBiDAF(nn.Module):
    def __init__(self, n_words, n_chars, cfg=None, word_vectors=None):
        super().__init__()
        self.cfg = cfg = {**DEFAULT_MODEL, **(cfg or {})}
        H, p = cfg["hidden"], cfg["dropout"]
        self.word_emb = nn.Embedding(n_words, cfg["word_dim"], padding_idx=0)
        if word_vectors is not None:
            self.word_emb.weight.data.copy_(word_vectors)
        emb_dim = cfg["word_dim"]
        if cfg["use_char_cnn"]:
            self.char_cnn = CharCNN(n_chars, cfg["char_dim"], cfg["char_filters"], cfg["char_widths"])
            emb_dim += self.char_cnn.out_dim
        if cfg["use_token_flags"]:
            emb_dim += cfg["n_flags"]
        self.highway = Highway(emb_dim, 2)
        self.drop = nn.Dropout(p)
        self.encoder = BiLSTM(emb_dim, H)
        self.w_sim = nn.Linear(6 * H, 1)           # trilinear: w^T [h; u; h∘u]
        g_dim = 8 * H if cfg["biattn_mode"] == "both" else 6 * H
        self.g_proj = nn.Linear(g_dim, 2 * H)
        if cfg["use_self_attn"]:
            self.self_attn = GatedSelfAttention(2 * H, cfg["attn_heads"], p)
        self.model1 = BiLSTM(2 * H, H, layers=2, dropout=p)
        self.model2 = BiLSTM(2 * H, H)
        self.start_head = nn.Linear(4 * H, 1)
        self.end_head = nn.Linear(4 * H, 1)

    # ---------------------------------------------------------------- layers
    def embed(self, words, chars, flags):
        parts = [self.word_emb(words)]
        if self.cfg["use_char_cnn"]:
            parts.append(self.char_cnn(chars))
        if self.cfg["use_token_flags"]:
            parts.append(flags)
        return self.highway(self.drop(torch.cat(parts, -1)))

    def similarity(self, h, u):
        """S_ij = w_h·h_i + w_u·u_j + w_hu·(h_i∘u_j), computed without materialising [h; u; h∘u]."""
        d = h.size(-1)
        w = self.w_sim.weight.squeeze(0)
        wh, wu, whu = w[:d], w[d:2 * d], w[2 * d:]
        return (h @ wh).unsqueeze(2) + (u @ wu).unsqueeze(1) + (h * whu) @ u.transpose(1, 2) + self.w_sim.bias

    def forward(self, batch, return_attn=False):
        tm, qm = batch["t_mask"], batch["q_mask"]
        h = self.drop(self.encoder(self.embed(batch["t_words"], batch["t_chars"], batch["t_flags"]), tm))
        u = self.drop(self.encoder(self.embed(batch["q_words"], batch["q_chars"], batch["q_flags"]), qm))

        S = self.similarity(h, u)                                        # (B, T, J)
        a = F.softmax(S.masked_fill(~qm.unsqueeze(1), NEG), dim=-1)      # C2Q weights
        c2q = a @ u
        parts = [h, c2q, h * c2q]
        b = None
        if self.cfg["biattn_mode"] == "both":
            smax = S.masked_fill(~qm.unsqueeze(1), NEG).max(-1).values    # (B, T)
            b = F.softmax(smax.masked_fill(~tm, NEG), dim=-1)            # Q2C weights
            q2c = (b.unsqueeze(1) @ h).expand_as(h)
            parts.append(h * q2c)
        x = self.drop(F.relu(self.g_proj(torch.cat(parts, -1))))

        sa = None
        if self.cfg["use_self_attn"]:
            x, sa = self.self_attn(x, tm)

        m1 = self.drop(self.model1(x, tm))
        m2 = self.drop(self.model2(m1, tm))
        start = self.start_head(torch.cat([x, m1], -1)).squeeze(-1).masked_fill(~tm, NEG)
        end = self.end_head(torch.cat([x, m2], -1)).squeeze(-1).masked_fill(~tm, NEG)
        out = dict(start_logits=start, end_logits=end)
        if return_attn:
            out.update(S=S, c2q=a, q2c=b, self_attn=sa)
        return out


# ------------------------------------------------------------------ loss / decode

def span_loss(out, batch, soft=True):
    ls = F.log_softmax(out["start_logits"], -1)
    le = F.log_softmax(out["end_logits"], -1)
    if soft:
        # soft cross-entropy against the marginal start/end distributions of the soft silver set
        return -(batch["soft_start"] * ls).sum(-1).mean() - (batch["soft_end"] * le).sum(-1).mean()
    return F.nll_loss(ls, batch["y_start"]) + F.nll_loss(le, batch["y_end"])


@torch.no_grad()
def decode(start_logits, end_logits, max_len=15):
    """argmax over i <= j < i + max_len of log p_s(i) + log p_e(j). Returns (B, 2) indices and scores."""
    ls = F.log_softmax(start_logits, -1)
    le = F.log_softmax(end_logits, -1)
    T = ls.size(1)
    scores = ls.unsqueeze(2) + le.unsqueeze(1)                           # (B, T, T)
    idx = torch.arange(T, device=ls.device)
    band = (idx.unsqueeze(0) >= idx.unsqueeze(1)) & (idx.unsqueeze(0) - idx.unsqueeze(1) < max_len)
    scores = scores.masked_fill(~band, NEG)
    flat = scores.view(scores.size(0), -1)
    best = flat.argmax(-1)
    return torch.stack([best // T, best % T], -1), flat.max(-1).values


In [ ]:
%%writefile src/fasttext_init.py
"""Optional in-domain word vectors: FastText (gensim) trained from scratch on the
*training split's* tweets and questions only. No external text and no
pretrained weights, so the main model is still trained from scratch on
TweetQA alone."""
import torch


def fasttext_vectors(wv, cfg, train_examples, epochs=20, seed=1):
    from gensim.models import FastText
    sents, seen = [], set()
    for ex in train_examples:
        if ex["tweet"] not in seen:                 # each tweet once
            seen.add(ex["tweet"])
            sents.append([t["word"] for t in ex["t_tokens"]])
        sents.append([t["word"] for t in ex["q_tokens"]])
    ft = FastText(vector_size=cfg["word_dim"], window=5, min_count=1, sg=1, epochs=epochs,
                  seed=seed, workers=1, min_n=3, max_n=5)
    ft.build_vocab(sents)
    ft.train(sents, total_examples=len(sents), epochs=epochs)
    W = torch.randn(len(wv), cfg["word_dim"]) * 0.1
    W[0] = 0.0
    for i, w in enumerate(wv.itos[2:], start=2):
        W[i] = torch.tensor(ft.wv[w])               # subword n-grams also give OOV-safe vectors
    return W


In [ ]:
%%writefile src/train.py
"""Training loop for TweetBiDAF: config dict -> train / early stop / checkpoint / runs.csv.

Usage (from A2/):
  python -m src.train --name main --seed 1                        # full run
  python -m src.train --name smoke --smoke                        # 200 examples, 2 epochs
  python -m src.train --name a3 --set use_self_attn=false --seed 2
"""
import argparse
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

try:
    from . import data as D
    from .evaluate import evaluate, fmt
    from .model import TweetBiDAF, decode, span_loss
    from .runlog import RESULTS_DIR, log_run
except ImportError:
    import data as D
    from evaluate import evaluate, fmt
    from model import TweetBiDAF, decode, span_loss
    from runlog import RESULTS_DIR, log_run

DEFAULT_TRAIN = dict(
    lr=1e-3, batch_size=32, epochs=30, patience=4, grad_clip=5.0, weight_decay=0.0,
    use_normaliser=True, tau=0.5, soft_tau=None, soft_spans=True, min_freq=2, word_init="random",
    max_span_len=15,
)
CKPT_DIR = Path(os.environ.get("A2_CKPT", RESULTS_DIR.parent / "checkpoints"))


def get_device():
    return "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def prepare_data(cfg, train_rows=None, val_rows=None, limit=None):
    """Preprocess and encode the train/val splits. Vocab comes from the training split only."""
    if train_rows is None:
        train_rows, val_rows = D.get_splits()
    if limit:
        train_rows, val_rows = train_rows[:limit], val_rows[:limit]
    prep = dict(use_normaliser=cfg["use_normaliser"], max_span_len=cfg["max_span_len"], tau=cfg["tau"],
                soft_tau=cfg.get("soft_tau"))
    tr, va = D.preprocess(train_rows, prep), D.preprocess(val_rows, prep)
    wv, cv = D.build_vocabs(tr, min_freq=cfg["min_freq"])
    D.encode(tr, wv, cv)
    D.encode(va, wv, cv)
    return tr, va, wv, cv


@torch.no_grad()
def predict(model, examples, device, batch_size=64, max_span_len=15):
    model.eval()
    preds, spans = [], []
    for batch in D.make_loader(examples, batch_size=batch_size):
        meta = batch["meta"]
        out = model(to_device(batch, device))
        idx, _ = decode(out["start_logits"], out["end_logits"], max_span_len)
        for ex, (i, j) in zip(meta, idx.tolist()):
            spans.append((i, j))
            preds.append(D.span_text(ex, i, j))
    return preds, spans


def build_model(cfg, wv, cv, word_vectors=None):
    mcfg = {k: cfg[k] for k in _default_model() if k in cfg}
    return TweetBiDAF(len(wv), len(cv), mcfg, word_vectors)


def train(cfg=None, seed=1, name="main", limit=None, data=None, log=print, save=True):
    """Train one model. Returns dict(best scores, history, checkpoint path, run_id)."""
    cfg = {**DEFAULT_TRAIN, **_default_model(), **(cfg or {})}
    seed_everything(seed)
    device = get_device()
    tr, va, wv, cv = data if data is not None else prepare_data(cfg, limit=limit)
    wvecs = None
    if cfg["word_init"] == "fasttext":
        try:
            from .fasttext_init import fasttext_vectors
        except ImportError:
            from fasttext_init import fasttext_vectors
        wvecs = fasttext_vectors(wv, cfg, tr, seed=seed)
    model = build_model(cfg, wv, cv, wvecs).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loader = D.make_loader(tr, batch_size=cfg["batch_size"], train=True, seed=seed)
    golds = [e["answers"] for e in va]
    run_id = f"{name}-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    log(f"[{run_id}] device={device} params={n_params:,} train_used={len(loader.dataset)}/{len(tr)} "
        f"val={len(va)} vocab={len(wv)} chars={len(cv)}")
    log(f"[{run_id}] config={json.dumps(cfg, sort_keys=True)}")

    best, best_epoch, bad, history, best_preds = None, -1, 0, [], []
    ckpt = CKPT_DIR / f"{run_id}.pt"
    t_start = time.time()
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for batch in loader:
            batch = to_device(batch, device)
            loss = span_loss(model(batch), batch, soft=cfg["soft_spans"])
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
            tot += loss.item() * batch["t_words"].size(0)
            n += batch["t_words"].size(0)
        train_time = time.time() - t0
        preds, spans = predict(model, va, device, max_span_len=cfg["max_span_len"])
        scores = evaluate(preds, golds)
        history.append(dict(epoch=epoch, train_loss=tot / max(1, n), epoch_time=train_time, **scores))
        improved = best is None or scores["rougeL"] > best["rougeL"]
        log(f"[{run_id}] epoch {epoch:2d} loss={tot / max(1, n):.4f} time={train_time:.1f}s val {fmt(scores)}"
            + (" *" if improved else ""))
        if improved:
            best, best_epoch, bad = scores, epoch, 0
            best_preds = [dict(row_id=e["row_id"], qid=e["qid"], pred=p, span=sp)
                          for e, p, sp in zip(va, preds, spans)]
            if save:
                CKPT_DIR.mkdir(parents=True, exist_ok=True)
                torch.save(dict(state_dict=model.state_dict(), cfg=cfg, seed=seed, epoch=epoch,
                                vocab=wv.itos, chars=cv.itos), ckpt)
        else:
            bad += 1
            if bad >= cfg["patience"]:
                log(f"[{run_id}] early stop at epoch {epoch} (best epoch {best_epoch})")
                break
    total = time.time() - t_start
    log(f"[{run_id}] best epoch {best_epoch}: val {fmt(best)} | total {total/60:.1f} min")
    if save:   # smoke tests (save=False) leave no files behind, so results/ only holds logged runs
        hist_dir = RESULTS_DIR / "history"
        hist_dir.mkdir(parents=True, exist_ok=True)
        (hist_dir / f"{run_id}.json").write_text(json.dumps(dict(cfg=cfg, seed=seed, history=history,
                                                                 best_epoch=best_epoch), indent=1))
        pred_dir = RESULTS_DIR / "preds"
        pred_dir.mkdir(parents=True, exist_ok=True)
        (pred_dir / f"{run_id}.json").write_text(json.dumps(best_preds))
        log_run(name, cfg, seed, "val", best, total, run_id=run_id,
                notes=f"best_epoch={best_epoch} params={n_params} limit={limit} device={device}"
                + (f":{torch.cuda.get_device_name(0)}" if device == "cuda" else ""))
    return dict(best=best, best_epoch=best_epoch, history=history, ckpt=str(ckpt), run_id=run_id,
                model=model, data=(tr, va, wv, cv))


def _default_model():
    try:
        from .model import DEFAULT_MODEL
    except ImportError:
        from model import DEFAULT_MODEL
    return dict(DEFAULT_MODEL)


def load_checkpoint(path, device=None):
    device = device or get_device()
    ck = torch.load(path, map_location="cpu", weights_only=False)
    wv, cv = D.Vocab({}, 1), D.Vocab({}, 1)
    wv.itos, cv.itos = ck["vocab"], ck["chars"]
    wv.stoi = {w: i for i, w in enumerate(wv.itos)}
    cv.stoi = {c: i for i, c in enumerate(cv.itos)}
    model = build_model(ck["cfg"], wv, cv)
    model.load_state_dict(ck["state_dict"])
    return model.to(device).eval(), ck["cfg"], wv, cv


def _parse_set(items):
    """Parse key=json_value overrides. Unknown keys and values whose type doesn't match the default
    (e.g. a bool flag given a string) are errors, so a typo can never silently change an experiment."""
    defaults = {**DEFAULT_TRAIN, **_default_model()}
    out = {}
    for it in items or []:
        k, v = it.split("=", 1)
        if k not in defaults:
            raise SystemExit(f"--set: unknown config key {k!r}")
        try:
            val = json.loads(v)
        except json.JSONDecodeError:
            val = v
        d = defaults[k]
        ok = (d is None or val is None or isinstance(val, type(d))
              or (isinstance(d, float) and isinstance(val, int) and not isinstance(val, bool))
              or (isinstance(d, tuple) and isinstance(val, list)))
        if isinstance(d, bool) and not isinstance(val, bool):
            ok = False
        if not ok:
            raise SystemExit(f"--set: {k}={v!r} has type {type(val).__name__}, expected {type(d).__name__}")
        out[k] = val
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="main")
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--smoke", action="store_true", help="200 examples, 2 epochs, no runs.csv entry")
    ap.add_argument("--set", nargs="*", help="config overrides key=json_value")
    args = ap.parse_args()
    cfg = _parse_set(args.set)
    if args.smoke:
        cfg.update(epochs=2, patience=99)
    res = train(cfg, seed=args.seed, name=args.name, limit=200 if args.smoke else None,
                log=lambda s: print(s, flush=True), save=not args.smoke)
    if args.smoke:
        h = res["history"]
        assert h[-1]["train_loss"] < h[0]["train_loss"], "smoke test: loss did not decrease"
        tr, va, wv, cv = res["data"]
        model = res["model"]
        batch = to_device(D.collate(va[:4]), get_device())
        with torch.no_grad():
            model.eval()
            o = model(batch, return_attn=True)
        B, T = batch["t_words"].shape
        J = batch["q_words"].shape[1]
        assert o["S"].shape == (B, T, J) and o["c2q"].shape == (B, T, J)
        if o["q2c"] is not None:
            assert o["q2c"].shape == (B, T)
        if o["self_attn"] is not None:
            assert o["self_attn"].shape == (B, T, T)
        assert torch.allclose(o["c2q"].sum(-1), torch.ones(B, T, device=o["c2q"].device), atol=1e-4)
        print(f"SMOKE OK: loss {h[0]['train_loss']:.3f} -> {h[-1]['train_loss']:.3f}; "
              f"attn shapes S{tuple(o['S'].shape)} self{tuple(o['self_attn'].shape) if o['self_attn'] is not None else None}")
    sys.exit(0)


In [ ]:
%%writefile src/baselines/b3_extractive_ft.py
"""B3: fine-tuned pretrained encoder (vinai/bertweet-base, fallback roberta-base)
with a span head, trained on the same silver spans as the main model.

Each of our word tokens is BPE-encoded separately. That keeps a subword -> word
map, so the predicted subword span maps back to our tokens and then to raw
tweet text through the same char offsets the main model uses. This works with
BERTweet's slow tokenizer, which has no offset mapping.

Input: <s> question </s></s> tweet </s>, truncated to max_len subwords
(the tweet is truncated first; this is rare at 128).
"""
import argparse
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))
from src import data as D  # noqa: E402
from src.evaluate import evaluate, fmt  # noqa: E402
from src.runlog import log_run  # noqa: E402
from src.train import get_device  # noqa: E402

B3_CFG = dict(model="vinai/bertweet-base", lr=3e-5, epochs=3, batch_size=16, max_len=128,
           warmup=0.1, weight_decay=0.01, max_answer_subwords=30)


def b3_word_piece(tok, t):
    raw = t["raw"]
    if t["kind"] == "mention":
        raw = "@USER" if "bertweet" in tok.name_or_path else raw
    elif t["kind"] == "url":
        raw = "HTTPURL" if "bertweet" in tok.name_or_path else raw
    ids = tok.encode(" " + raw if "roberta" in tok.name_or_path else raw, add_special_tokens=False)
    return ids or [tok.unk_token_id]


def b3_featurise(tok, ex, max_len):
    q_ids = [i for t in ex["q_tokens"] for i in b3_word_piece(tok, t)][:40]
    ids = [tok.cls_token_id] + q_ids + [tok.sep_token_id, tok.sep_token_id]
    word_of = [-1] * len(ids)
    budget = max_len - len(ids) - 1
    for w, t in enumerate(ex["t_tokens"]):
        p = b3_word_piece(tok, t)
        if len(p) > budget:
            break
        ids += p
        word_of += [w] * len(p)
        budget -= len(p)
    ids.append(tok.sep_token_id)
    word_of.append(-1)
    ys = ye = -1
    if ex["best"] is not None:
        i, j = ex["best"]
        if i in word_of and j in word_of:
            ys = word_of.index(i)
            ye = len(word_of) - 1 - word_of[::-1].index(j)
    return dict(ids=ids, word_of=word_of, ys=ys, ye=ye)


def b3_batches(feats, bs, shuffle, rng, pad_id):
    order = list(range(len(feats)))
    if shuffle:
        rng.shuffle(order)
    for k in range(0, len(order), bs):
        fs = [feats[i] for i in order[k:k + bs]]
        T = max(len(f["ids"]) for f in fs)
        ids = torch.full((len(fs), T), pad_id, dtype=torch.long)
        att = torch.zeros(len(fs), T, dtype=torch.long)
        ctx = torch.zeros(len(fs), T, dtype=torch.bool)
        for b, f in enumerate(fs):
            ids[b, :len(f["ids"])] = torch.tensor(f["ids"])
            att[b, :len(f["ids"])] = 1
            ctx[b, :len(f["ids"])] = torch.tensor([w >= 0 for w in f["word_of"]])
        yield order[k:k + bs], ids, att, ctx, torch.tensor([f["ys"] for f in fs]), torch.tensor([f["ye"] for f in fs])


class SpanModel(torch.nn.Module):
    def __init__(self, name):
        super().__init__()
        from transformers import AutoModel
        self.enc = AutoModel.from_pretrained(name)
        self.head = torch.nn.Linear(self.enc.config.hidden_size, 2)

    def forward(self, ids, att, ctx):
        h = self.enc(input_ids=ids, attention_mask=att).last_hidden_state
        s, e = self.head(h).unbind(-1)
        return s.masked_fill(~ctx, -1e4), e.masked_fill(~ctx, -1e4)


@torch.no_grad()
def b3_predict(model, exs, feats, device, cfg, pad_id):
    model.eval()
    preds = [""] * len(exs)
    for idx, ids, att, ctx, _, _ in b3_batches(feats, 64, False, None, pad_id):
        s, e = model(ids.to(device), att.to(device), ctx.to(device))
        ls, le = F.log_softmax(s, -1), F.log_softmax(e, -1)
        T = ls.size(1)
        sc = ls.unsqueeze(2) + le.unsqueeze(1)
        r = torch.arange(T, device=device)
        band = (r[None] >= r[:, None]) & (r[None] - r[:, None] < cfg["max_answer_subwords"])
        flat = sc.masked_fill(~band, -1e9).view(len(idx), -1).argmax(-1).tolist()
        for k, f in zip(idx, flat):
            i, j = feats[k]["word_of"][f // T], feats[k]["word_of"][f % T]
            if i >= 0 and j >= i:
                preds[k] = D.span_text(exs[k], i, j)
    return preds


def b3_run(cfg, seed=1, split="val", limit=None, log=print):
    from transformers import AutoTokenizer, get_linear_schedule_with_warmup
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = get_device()
    try:
        tok = AutoTokenizer.from_pretrained(cfg["model"], use_fast=False)
    except Exception as e:  # noqa: BLE001
        log(f"could not load {cfg['model']} ({e}); falling back to roberta-base")
        cfg = {**cfg, "model": "roberta-base"}
        tok = AutoTokenizer.from_pretrained(cfg["model"])
    train_rows, val_rows = D.get_splits()
    eval_rows = val_rows if split == "val" else D.load_test()
    if limit:
        train_rows, eval_rows = train_rows[:limit], eval_rows[:limit]
    tr, ev = D.preprocess(train_rows), D.preprocess(eval_rows)
    ftr = [b3_featurise(tok, e, cfg["max_len"]) for e in tr if e["trainable"]]
    ftr = [f for f in ftr if f["ys"] >= 0]
    fev = [b3_featurise(tok, e, cfg["max_len"]) for e in ev]
    model = SpanModel(cfg["model"]).to(device)
    nd = ["bias", "LayerNorm.weight", "norm"]
    groups = [dict(params=[p for n, p in model.named_parameters() if not any(x in n for x in nd)],
                   weight_decay=cfg["weight_decay"]),
              dict(params=[p for n, p in model.named_parameters() if any(x in n for x in nd)], weight_decay=0.0)]
    opt = torch.optim.AdamW(groups, lr=cfg["lr"])
    steps = cfg["epochs"] * ((len(ftr) + cfg["batch_size"] - 1) // cfg["batch_size"])
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup"] * steps), steps)
    rng = random.Random(seed)
    log(f"B3 {cfg['model']} device={device} train_feats={len(ftr)} eval={len(fev)} steps={steps}")
    t0 = time.time()
    for ep in range(cfg["epochs"]):
        model.train()
        tot = n = 0
        for _, ids, att, ctx, ys, ye in b3_batches(ftr, cfg["batch_size"], True, rng, tok.pad_token_id):
            ids, att, ctx, ys, ye = (x.to(device) for x in (ids, att, ctx, ys, ye))
            s, e = model(ids, att, ctx)
            loss = F.cross_entropy(s, ys) + F.cross_entropy(e, ye)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tot += loss.item() * len(ys); n += len(ys)
        preds = b3_predict(model, ev, fev, device, cfg, tok.pad_token_id)
        s = evaluate(preds, [e["answers"] for e in ev])
        log(f"B3 epoch {ep+1} loss={tot/n:.4f} {split} {fmt(s)} ({(time.time()-t0)/60:.1f} min)")
    return s, preds, time.time() - t0, cfg, ev


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--split", default="val")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--model", default=B3_CFG["model"])
    a = ap.parse_args()
    cfg = {**B3_CFG, "model": a.model}
    if a.smoke:
        cfg["epochs"] = 1
    s, preds, tt, cfg, ev = b3_run(cfg, a.seed, a.split, limit=200 if a.smoke else None,
                                log=lambda x: print(x, flush=True))
    if not a.smoke:
        rid = log_run("B3_" + cfg["model"].split("/")[-1], cfg, a.seed, a.split, s, tt,
                      notes="fine-tuned pretrained encoder + span head on silver spans (final epoch)")
        out = ROOT / "results" / "preds"
        out.mkdir(parents=True, exist_ok=True)
        (out / f"{rid}.json").write_text(json.dumps([dict(qid=e["qid"], row_id=e["row_id"], pred=p)
                                                     for e, p in zip(ev, preds)]))
    print("DONE", fmt(s), flush=True)


In [ ]:
%%writefile src/baselines/b4_t5_ft.py
"""B4: fine-tuned generative seq2seq (t5-small; google/flan-t5-base optional).

Input "question: <q> context: <tweet>" -> the gold answer string. Unlike the
extractive models it can produce answers that are not in the tweet
(about 35% of TweetQA). Beam search 4, max 16 new tokens.
"""
import argparse
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))
from src import data as D  # noqa: E402
from src.evaluate import evaluate, fmt  # noqa: E402
from src.runlog import log_run  # noqa: E402
from src.train import get_device  # noqa: E402

B4_CFG = dict(model="t5-small", lr=3e-4, epochs=5, batch_size=16, max_src=160, max_tgt=24,
           beams=4, max_new_tokens=16, weight_decay=0.0)


def b4_src_text(r):
    return f"question: {r['question']} context: {r['tweet']}"


@torch.no_grad()
def b4_generate(model, tok, rows, device, cfg, bs=64):
    model.eval()
    out = []
    for k in range(0, len(rows), bs):
        enc = tok([b4_src_text(r) for r in rows[k:k + bs]], return_tensors="pt", padding=True,
                  truncation=True, max_length=cfg["max_src"]).to(device)
        gen = model.generate(**enc, num_beams=cfg["beams"], max_new_tokens=cfg["max_new_tokens"])
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return [o.strip() for o in out]


def b4_run(cfg, seed=1, split="val", limit=None, log=print):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = get_device()
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg["model"]).to(device)
    train_rows, val_rows = D.get_splits()
    eval_rows = val_rows if split == "val" else D.load_test()
    if limit:
        train_rows, eval_rows = train_rows[:limit], eval_rows[:limit]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    steps = cfg["epochs"] * ((len(train_rows) + cfg["batch_size"] - 1) // cfg["batch_size"])
    sched = get_linear_schedule_with_warmup(opt, 0, steps)
    rng = random.Random(seed)
    log(f"B4 {cfg['model']} device={device} train={len(train_rows)} eval={len(eval_rows)} steps={steps}")
    t0 = time.time()
    s, preds = None, None
    for ep in range(cfg["epochs"]):
        model.train()
        order = list(range(len(train_rows)))
        rng.shuffle(order)
        tot = n = 0
        for k in range(0, len(order), cfg["batch_size"]):
            rows = [train_rows[i] for i in order[k:k + cfg["batch_size"]]]
            enc = tok([b4_src_text(r) for r in rows], return_tensors="pt", padding=True, truncation=True,
                      max_length=cfg["max_src"]).to(device)
            lab = tok([r["answers"][0] for r in rows], return_tensors="pt", padding=True, truncation=True,
                      max_length=cfg["max_tgt"]).input_ids.to(device)
            lab[lab == tok.pad_token_id] = -100
            loss = model(**enc, labels=lab).loss
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tot += loss.item() * len(rows); n += len(rows)
        preds = b4_generate(model, tok, eval_rows, device, cfg)
        s = evaluate(preds, [r["answers"] for r in eval_rows])
        log(f"B4 epoch {ep+1} loss={tot/n:.4f} {split} {fmt(s)} ({(time.time()-t0)/60:.1f} min)")
    return s, preds, time.time() - t0, eval_rows


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--split", default="val")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--model", default=B4_CFG["model"])
    ap.add_argument("--epochs", type=int, default=B4_CFG["epochs"])
    a = ap.parse_args()
    cfg = {**B4_CFG, "model": a.model, "epochs": 1 if a.smoke else a.epochs}
    s, preds, tt, rows = b4_run(cfg, a.seed, a.split, limit=200 if a.smoke else None,
                             log=lambda x: print(x, flush=True))
    if not a.smoke:
        rid = log_run("B4_" + cfg["model"].split("/")[-1], cfg, a.seed, a.split, s, tt,
                      notes="fine-tuned seq2seq, beam 4 (final epoch)")
        out = ROOT / "results" / "preds"
        out.mkdir(parents=True, exist_ok=True)
        (out / f"{rid}.json").write_text(json.dumps([dict(qid=r["qid"], row_id=r["row_id"], pred=p)
                                                     for r, p in zip(rows, preds)]))
    print("DONE", fmt(s), flush=True)


In [ ]:
%%writefile src/baselines/b5_llm.py
"""B5: open instruct LLM (Qwen/Qwen2.5-1.5B-Instruct) with a fixed few-shot prompt,
greedy decoding, output stripped to a short answer. No fine-tuning.

The prompt is recorded verbatim in PROMPT_SYSTEM / the few-shot block below. The
few-shot examples are fixed training-split rows chosen by row id.
"""
import argparse
import json
import re
import sys
import time
from pathlib import Path

import torch

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))
from src import data as D  # noqa: E402
from src.evaluate import evaluate, fmt  # noqa: E402
from src.runlog import log_run  # noqa: E402
from src.train import get_device  # noqa: E402

B5_CFG = dict(model="Qwen/Qwen2.5-1.5B-Instruct", shots=4, max_new_tokens=16, batch_size=16)
PROMPT_SYSTEM = ("You answer questions about a tweet. Reply with only the short answer "
                 "(usually a few words copied from the tweet), with no explanation.")
SHOT_QWORDS = ("who", "what", "when", "how")   # one fixed train example per question type


def pick_shots(train_rows, k):
    shots = []
    for w in SHOT_QWORDS[:k]:
        for r in train_rows:        # first train row (fixed order) with this q-word and a short extractive answer
            a = r["answers"][0]
            if r["question"].lower().startswith(w) and a.lower() in r["tweet"].lower() and len(a.split()) <= 4:
                shots.append(r)
                break
    return shots


def b5_messages(r, shots):
    msgs = [dict(role="system", content=PROMPT_SYSTEM)]
    for s in shots:
        msgs.append(dict(role="user", content=f"Tweet: {s['tweet']}\nQuestion: {s['question']}"))
        msgs.append(dict(role="assistant", content=s["answers"][0]))
    msgs.append(dict(role="user", content=f"Tweet: {r['tweet']}\nQuestion: {r['question']}"))
    return msgs


def b5_clean(text):
    text = text.strip().split("\n")[0]
    text = re.sub(r"^(answer|a)\s*:\s*", "", text, flags=re.I)
    return text.strip().strip('"').rstrip(".")


def b5_run(cfg, split="val", limit=None, log=print):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    torch.manual_seed(0)
    device = get_device()
    tok = AutoTokenizer.from_pretrained(cfg["model"], padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(cfg["model"], dtype=torch.float16
                                                 if device != "cpu" else torch.float32).to(device).eval()
    train_rows, val_rows = D.get_splits()
    rows = val_rows if split == "val" else D.load_test()
    if limit:
        rows = rows[:limit]
    shots = pick_shots(train_rows, cfg["shots"])
    log(f"B5 {cfg['model']} device={device} eval={len(rows)} shots={[s['row_id'] for s in shots]}")
    log("PROMPT (verbatim, last user turn is the target):\n" + json.dumps(b5_messages(rows[0], shots), indent=1))
    t0 = time.time()
    preds = []
    for k in range(0, len(rows), cfg["batch_size"]):
        chunk = rows[k:k + cfg["batch_size"]]
        texts = [tok.apply_chat_template(b5_messages(r, shots), tokenize=False, add_generation_prompt=True)
                 for r in chunk]
        enc = tok(texts, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, do_sample=False, max_new_tokens=cfg["max_new_tokens"],
                                 pad_token_id=tok.eos_token_id)
        preds += [b5_clean(t) for t in tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
        if k % (cfg["batch_size"] * 10) == 0:
            log(f"  {k + len(chunk)}/{len(rows)} ({(time.time()-t0)/60:.1f} min)")
    s = evaluate(preds, [r["answers"] for r in rows])
    return s, preds, time.time() - t0, rows, shots


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="val")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--model", default=B5_CFG["model"])
    ap.add_argument("--shots", type=int, default=B5_CFG["shots"])
    a = ap.parse_args()
    cfg = {**B5_CFG, "model": a.model, "shots": a.shots}
    s, preds, tt, rows, shots = b5_run(cfg, a.split, limit=32 if a.smoke else None, log=lambda x: print(x, flush=True))
    for r, p in list(zip(rows, preds))[:8]:
        print(f"  Q: {r['question']} | gold: {r['answers']} | pred: {p}")
    if not a.smoke:
        cfg = {**cfg, "prompt_system": PROMPT_SYSTEM, "shot_row_ids": [x["row_id"] for x in shots]}
        rid = log_run("B5_" + cfg["model"].split("/")[-1] + f"_{cfg['shots']}shot", cfg, 0, a.split, s, tt,
                      notes="few-shot instruct LLM, greedy, no fine-tuning")
        out = ROOT / "results" / "preds"
        out.mkdir(parents=True, exist_ok=True)
        (out / f"{rid}.json").write_text(json.dumps([dict(qid=r["qid"], row_id=r["row_id"], pred=p)
                                                     for r, p in zip(rows, preds)]))
    print("DONE", fmt(s), flush=True)


In [ ]:
%%writefile scripts/run_queue.py
"""Run a list of training jobs as subprocesses (N in parallel), one log per job.

  python scripts/run_queue.py jobs.json --parallel 2 --tag stage4

jobs.json: [{"name": "lr5e-4", "seed": 1, "set": {"lr": 5e-4}}, ...]
Each job must pass a smoke test first unless "--no-smoke" is given.
"""
import argparse
import json
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
LOGS = ROOT / "results" / "logs"


def cmd(job, smoke=False):
    c = [sys.executable, "-m", "src.train", "--name", job["name"], "--seed", str(job.get("seed", 1))]
    if smoke:
        c.append("--smoke")
    if job.get("set"):
        c += ["--set"] + [f"{k}={json.dumps(v)}" for k, v in job["set"].items()]
    return c


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("jobs")
    ap.add_argument("--parallel", type=int, default=1)
    ap.add_argument("--tag", default="queue")
    ap.add_argument("--no-smoke", action="store_true")
    ap.add_argument("--gpus", type=int, default=0, help="spread parallel jobs over this many CUDA GPUs")
    a = ap.parse_args()
    jobs = json.loads(Path(a.jobs).read_text())
    LOGS.mkdir(parents=True, exist_ok=True)
    env = {"PYTHONUNBUFFERED": "1", **__import__("os").environ}
    if not a.no_smoke:
        seen = set()
        for j in jobs:
            key = json.dumps(j.get("set", {}), sort_keys=True)
            if key in seen:
                continue
            seen.add(key)
            r = subprocess.run(cmd(j, smoke=True), cwd=ROOT, env=env, capture_output=True, text=True)
            ok = r.returncode == 0 and "SMOKE OK" in r.stdout
            print(f"[{a.tag}] smoke {j['name']}: {'OK' if ok else 'FAILED'}", flush=True)
            if not ok:
                print(r.stdout[-2000:], r.stderr[-3000:], flush=True)
                sys.exit(1)
    pending, running = list(jobs), []
    while pending or running:
        while pending and len(running) < a.parallel:
            j = pending.pop(0)
            log = LOGS / f"{a.tag}_{j['name']}_s{j.get('seed', 1)}.log"
            f = open(log, "w")
            jenv = dict(env)
            if a.gpus:
                load = [sum(1 for r in running if r[4] == g) for g in range(a.gpus)]
                gpu = load.index(min(load))                     # least-loaded GPU
                jenv["CUDA_VISIBLE_DEVICES"] = str(gpu)
            else:
                gpu = None
            p = subprocess.Popen(cmd(j), cwd=ROOT, env=jenv, stdout=f, stderr=subprocess.STDOUT)
            running.append((p, j, f, time.time(), gpu))
            print(f"[{a.tag}] started {j['name']} s{j.get('seed', 1)} -> {log.name}", flush=True)
        time.sleep(5)
        for item in running[:]:
            p, j, f, t0, _ = item
            if p.poll() is not None:
                f.close()
                running.remove(item)
                print(f"[{a.tag}] finished {j['name']} s{j.get('seed', 1)} rc={p.returncode} "
                      f"({(time.time()-t0)/60:.1f} min)", flush=True)
    print(f"[{a.tag}] ALL DONE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
Path('results/FROZEN_CONFIG.json').write_text('''{
 "attn_heads": 4,
 "batch_size": 32,
 "biattn_mode": "both",
 "char_dim": 32,
 "char_filters": 50,
 "char_widths": [
  3,
  4,
  5
 ],
 "dropout": 0.2,
 "epochs": 30,
 "grad_clip": 5.0,
 "hidden": 128,
 "lr": 0.0005,
 "max_span_len": 15,
 "min_freq": 2,
 "n_flags": 8,
 "patience": 4,
 "soft_spans": true,
 "tau": 0.5,
 "use_char_cnn": true,
 "use_normaliser": true,
 "use_self_attn": true,
 "use_token_flags": true,
 "weight_decay": 0.0,
 "word_dim": 100,
 "word_init": "fasttext"
}''')
MAIN_JOBS = [{'name': 'cand_lr5e-4_ft', 'seed': 1, 'set': {'lr': 0.0005, 'word_init': 'fasttext'}}, {'name': 'cand_lr5e-4_ft', 'seed': 2, 'set': {'lr': 0.0005, 'word_init': 'fasttext'}}, {'name': 'cand_lr5e-4_ft', 'seed': 3, 'set': {'lr': 0.0005, 'word_init': 'fasttext'}}, {'name': 'A1_no_char', 'seed': 1, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_char_cnn': False}}, {'name': 'A1_no_char', 'seed': 2, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_char_cnn': False}}, {'name': 'A1_no_char', 'seed': 3, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_char_cnn': False}}, {'name': 'A2_no_norm_flags', 'seed': 1, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_normaliser': False, 'use_token_flags': False}}, {'name': 'A2_no_norm_flags', 'seed': 2, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_normaliser': False, 'use_token_flags': False}}, {'name': 'A2_no_norm_flags', 'seed': 3, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_normaliser': False, 'use_token_flags': False}}, {'name': 'A3_no_selfattn', 'seed': 1, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_self_attn': False}}, {'name': 'A3_no_selfattn', 'seed': 2, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_self_attn': False}}, {'name': 'A3_no_selfattn', 'seed': 3, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_self_attn': False}}, {'name': 'A4_c2q_only', 'seed': 1, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'biattn_mode': 'c2q'}}, {'name': 'A4_c2q_only', 'seed': 2, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'biattn_mode': 'c2q'}}, {'name': 'A4_c2q_only', 'seed': 3, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'biattn_mode': 'c2q'}}, {'name': 'A5_single_span', 'seed': 1, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'soft_spans': False}}, {'name': 'A5_single_span', 'seed': 2, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'soft_spans': False}}, {'name': 'A5_single_span', 'seed': 3, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'soft_spans': False}}, {'name': 'B2_vanilla', 'seed': 1, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_char_cnn': False, 'use_normaliser': False, 'use_token_flags': False, 'use_self_attn': False, 'biattn_mode': 'c2q', 'soft_spans': False}}, {'name': 'B2_vanilla', 'seed': 2, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_char_cnn': False, 'use_normaliser': False, 'use_token_flags': False, 'use_self_attn': False, 'biattn_mode': 'c2q', 'soft_spans': False}}, {'name': 'B2_vanilla', 'seed': 3, 'set': {'lr': 0.0005, 'word_init': 'fasttext', 'use_char_cnn': False, 'use_normaliser': False, 'use_token_flags': False, 'use_self_attn': False, 'biattn_mode': 'c2q', 'soft_spans': False}}]
STAGE34_JOBS = [{'name': 'main_v1', 'seed': 1, 'set': {}}, {'name': 'sw_base', 'seed': 2, 'set': {}}, {'name': 'sw_base', 'seed': 3, 'set': {}}, {'name': 'sw_lr5e-4', 'seed': 1, 'set': {'lr': 0.0005}}, {'name': 'sw_h96', 'seed': 1, 'set': {'hidden': 96}}, {'name': 'sw_h192', 'seed': 1, 'set': {'hidden': 192}}, {'name': 'sw_do0.3', 'seed': 1, 'set': {'dropout': 0.3}}, {'name': 'sw_L10', 'seed': 1, 'set': {'max_span_len': 10}}, {'name': 'sw_L20', 'seed': 1, 'set': {'max_span_len': 20}}, {'name': 'sw_fasttext', 'seed': 1, 'set': {'word_init': 'fasttext'}}, {'name': 'sw_lr5e-4', 'seed': 2, 'set': {'lr': 0.0005}}, {'name': 'sw_lr5e-4', 'seed': 3, 'set': {'lr': 0.0005}}]
print(len(MAIN_JOBS), 'main jobs;', len(STAGE34_JOBS), 'optional stage-3/4 jobs')

## Step 2 — Run the experiment queue

In [ ]:
# ---------------- helper: run a command, stream its output, keep a log ----------------
def sh(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    f = open(log, "w") if log else None
    for line in p.stdout:
        if f: f.write(line); f.flush()
        if "Warning" not in line and "it/s]" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if f: f.close()
    print("exit code", rc)
    return rc

# smoke test (200 examples, 2 epochs, checks loss decreases and attention shapes)
assert sh([sys.executable, "-m", "src.train", "--name", "smoke", "--smoke"]) == 0


In [ ]:
if RUN_MAIN:
    Path("results/gpu_main_jobs.json").write_text(json.dumps(MAIN_JOBS))
    t0 = time.time()
    sh([sys.executable, "scripts/run_queue.py", "results/gpu_main_jobs.json", "--parallel", str(2 * N_GPU),
        "--gpus", str(N_GPU), "--tag", "gpu_main"], log="results/logs/gpu_main_queue.log")
    print(f"gpu_main done in {(time.time() - t0) / 60:.1f} min")


In [ ]:
if RUN_STAGE34:
    Path("results/gpu_stage34_jobs.json").write_text(json.dumps(STAGE34_JOBS))
    t0 = time.time()
    sh([sys.executable, "scripts/run_queue.py", "results/gpu_stage34_jobs.json", "--parallel", str(2 * N_GPU),
        "--gpus", str(N_GPU), "--tag", "gpu_stage34"], log="results/logs/gpu_stage34_queue.log")
    print(f"gpu_stage34 done in {(time.time() - t0) / 60:.1f} min")


In [ ]:
if RUN_BASELINES:
    sh([sys.executable, "src/baselines/b3_extractive_ft.py", "--seed", "1"], log="results/logs/stage6_b3_bertweet_s1.log")
    sh([sys.executable, "src/baselines/b4_t5_ft.py", "--seed", "1"], log="results/logs/stage6_b4_t5small_s1.log")
    sh([sys.executable, "src/baselines/b5_llm.py"], log="results/logs/stage6_b5_qwen1.5b.log")


In [ ]:
# ---------------- summary of everything logged in this session ----------------
import csv, statistics
rows = list(csv.DictReader(open("results/runs.csv"))) if Path("results/runs.csv").exists() else []
print(len(rows), "runs logged in this session")
groups = {}
for r in rows:
    groups.setdefault((r["model"], r["split"]), []).append(r)
for (m, split), rs in groups.items():
    vals = {k: [float(r[k]) for r in rs] for k in ("bleu1", "meteor", "rougeL", "em", "f1")}
    print(f"{m:26s} {split} n={len(rs)} " + " ".join(
        f"{k}={statistics.mean(v):.2f}" + (f"±{statistics.stdev(v):.2f}" if len(v) > 1 else "") for k, v in vals.items()))


In [ ]:
# ---------------- pack results + checkpoints into one zip ----------------
out = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if IN_COLAB else WORK.parent
zip_base = out / "a2_gpu_results"
import zipfile
# keep only the checkpoints the analysis and the final test evaluation need;
# every run's logs, per-epoch history and validation predictions are kept
KEEP_CKPT = ('cand_lr5e-4_ft-', 'A3_no_selfattn-', 'A5_single_span-', 'B2_vanilla-')
with zipfile.ZipFile(f"{zip_base}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in (WORK / "results").rglob("*"):
        if p.is_file():
            z.write(p, Path("gpu") / p.relative_to(WORK))
    for p in (WORK / "checkpoints").glob("*.pt"):
        if p.name.startswith(KEEP_CKPT):
            z.write(p, Path("gpu") / p.relative_to(WORK), compress_type=zipfile.ZIP_STORED)
print(f"{zip_base}.zip", round(Path(f"{zip_base}.zip").stat().st_size / 1e6, 1), "MB")
if ON_KAGGLE:
    print("files in /kaggle/working (this is what Kaggle saves):",
          [(q.name, round(q.stat().st_size / 1e6, 1)) for q in Path("/kaggle/working").iterdir()])
if IN_COLAB:
    from google.colab import files
    files.download(f"{zip_base}.zip")
